In [1]:
# 개와 고양이 이진분류 예제를 EfficientNet을 이용해서 구현
# tensorflow keras에서는 efficientnet을 공식적으로 지원하고 있어요!

# EfficientNet 은 버전이 여러개가 있어요!
# EfficientNetB0 ~ B7 까지 버전이 있어요
# B0 : 가장 간단한 모델이고 빠르게 학습시킬 수 있어요!
# 일반적으로 B3~B4를 사용. B5이상은 시간이 오래걸리지만 정확도가 높아요!
# EfficientNetV2B0~B3까지 개량된 모델도 있어요
# 내가 가지고 있는 이미지의 크기와 살짝 연관성이 있어요!
# EfficientNet 사용할 때 입력이미지의 크기를 고정시켜서 사용하는게 좋아요
# 우리는 EfficientNetB4를 사용할거기 때문에 입력이미지의 크기는 380X380으로 설정할거에요!

In [2]:
# 모듈 import
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB4
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Flatten, Dense, Dropout, GlobalAveragePooling2D, Input
from tensorflow.keras.layers import BatchNormalization, Activation
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.optimizers import SGD, RMSprop, Adam
from tensorflow.keras import mixed_precision 
# Float16과 float32를 적절히 섞어서 학습속도를 높이고 GPU의 메모리 사용량을 줄일 수 있어요(연산속도가 빨라진다)
mixed_precision.set_global_policy('mixed_float16')
# 이렇게 하면 모든 연산이 float16으로 계산되고 맨 마지막 output layer만 float32로 사용

INFO:tensorflow:Mixed precision compatibility check (mixed_float16): OK
Your GPU will likely run quickly with dtype policy mixed_float16 as it has compute capability of at least 7.0. Your GPU: NVIDIA GeForce RTX 3070 Ti Laptop GPU, compute capability 8.6


In [3]:
train_dir = './data/cat_dog_small/train'
validation_dir = './data/cat_dog_small/validation'
test_dir = './data/cat_dog_small/test'

In [4]:
# Parameter 설정
IMAGE_SIZE = 380
BATCH_SIZE = 64 # OOM 오류가 발생하면 숫자를 줄여야 해요!
LEARNING_RATE = 5e-5  
# EfficientNet은 상당히 민감한 모델이라 learning_rate에 영향을 많이 받아 이 값이 크면 불안정해져요. 일반적으로 사용하는 값보다 작게 잡는게 좋아요

In [5]:
# ImageDataGenerator 생성
train_datagen = ImageDataGenerator(preprocessing_function=preprocess_input,
                                  rotation_range=25,
                                  width_shift_range=0.1,
                                  height_shift_range=0.1,
                                  zoom_range=0.2,
                                  horizontal_flip=True,
                                  fill_mode='nearest')
validation_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)
test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

In [6]:
# ImageDataGenerator 설정
train_generator = train_datagen.flow_from_directory(
    train_dir,
    classes=['cats','dogs'],
    target_size=(IMAGE_SIZE,IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary'
)
validation_generator = validation_datagen.flow_from_directory(
    validation_dir,
    classes=['cats','dogs'],
    target_size=(IMAGE_SIZE,IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary'
)
test_generator = test_datagen.flow_from_directory(
    test_dir,
    classes=['cats','dogs'],
    target_size=(IMAGE_SIZE,IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

Found 2000 images belonging to 2 classes.
Found 1000 images belonging to 2 classes.
Found 1000 images belonging to 2 classes.


In [7]:
# model

pretrained_network = EfficientNetB4(weights='imagenet',
                                    include_top=False,
                                    input_shape=(IMAGE_SIZE,IMAGE_SIZE,3))

# pretrained_network.trainable = False  
# 이렇게 하는게 맞긴한데.. 간혹 동결이 풀리는 경우가 있어요.. .summary()로 확인 가능
for layer in pretrained_network.layers:
    layer.trainable = False

model = Sequential()

model.add(pretrained_network)
# model.add(Flatten()) 이거를 사용하면 12*12*XXX이라 데이터양이 많아져 대신 GlobalAveragePooling2D 사용
model.add(GlobalAveragePooling2D()) # 1*XXX 픽셀의 데이터를 줄일 수 있다! 성능도 up!
model.add(Dense(units=64))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(Dropout(rate=0.3))
model.add(Dense(units=1,
               activation='sigmoid'))

model.compile(optimizer=Adam(learning_rate=LEARNING_RATE),
             loss='binary_crossentropy',
             metrics=['accuracy'])

In [8]:
cp_callback = ModelCheckpoint(filepath='./efficientnet_weights.h5',
                             save_best_only=True,
                             save_weights_only=True,
                             monitor='val_accuracy',
                             verbose=1)

es_callback = EarlyStopping(monitor='val_loss',
                           patience=5,
                           verbose=1)

In [9]:
# model 1차 학습
model.fit(train_generator,
         steps_per_epoch=len(train_generator),
         epochs=20,
         validation_data=validation_generator,
         validation_steps=len(validation_generator),
         callbacks=[es_callback, cp_callback],
         verbose=1)

Epoch 1/20
32/32 [==============================] - ETA: 0s - loss: 0.5380 - accuracy: 0.7315  
Epoch 1: val_accuracy improved from -inf to 0.91500, saving model to ./efficientnet_weights.h5
32/32 [==============================] - 46s 1s/step - loss: 0.5380 - accuracy: 0.7315 - val_loss: 0.4883 - val_accuracy: 0.9150
Epoch 2/20
32/32 [==============================] - ETA: 0s - loss: 0.2493 - accuracy: 0.9335 
Epoch 2: val_accuracy improved from 0.91500 to 0.97000, saving model to ./efficientnet_weights.h5
32/32 [==============================] - 34s 1s/step - loss: 0.2493 - accuracy: 0.9335 - val_loss: 0.3530 - val_accuracy: 0.9700
Epoch 3/20
32/32 [==============================] - ETA: 0s - loss: 0.1696 - accuracy: 0.9690 
Epoch 3: val_accuracy improved from 0.97000 to 0.98800, saving model to ./efficientnet_weights.h5
32/32 [==============================] - 35s 1s/step - loss: 0.1696 - accuracy: 0.9690 - val_loss: 0.2630 - val_accuracy: 0.9880
Epoch 4/20
32/32 [==================

In [11]:
# Fine Tuning

for layer in pretrained_network.layers:
    layer.trainable = True

for layer in pretrained_network.layers[:-30]:
    layer.trainable = False    

In [ ]:
# 반드시 다시 compile 작업을 해야해요
# 러닝레이트는 줄여서 사용해야 하구요
# 옵티마이저도 새롭게 생성해서 사용해야 해요

# model 재설정
model.compile(optimizer=Adam(learning_rate=LEARNING_RATE*0.1),
             loss='binary_crossentropy',
             metrics=['accuracy'])

# model 1차 학습
model.fit(train_generator,
         steps_per_epoch=len(train_generator),
         epochs=20,
         validation_data=validation_generator,
         validation_steps=len(validation_generator),
         callbacks=[es_callback, cp_callback],
         verbose=1)